# Stage 1 Optuna: constrained median objective без mixed set + TS confirmation

Notebook запускает Optuna только для 8 feature-set candidates, прошедших default screening:

- 4 алгоритма × 2 method groups (`corr`, `hybrid`);
- без `mixed` feature set;
- objective без субъективных весов: **constrained median-of-medians return**;
- hard constraint по торговой активности: `MIN_TOTAL_TRADES = 250`;
- для Thompson Sampling после Optuna выполняется confirmation: **top-3 trials × 25 fresh seeds**;
- локальное сохранение результатов в папку проекта, без SQLite / внешней базы данных;
- можно запускать разные алгоритмы на разных компьютерах через `RUN_PRESET`.

Рекомендуемое разделение нагрузки:

- **Компьютер A:** `discounted_lints`, `discounted_linucb`
- **Компьютер B:** `sw_lints`, `sw_linucb`

При `N_TRIALS=40`, `TS_SEEDS_PER_TRIAL=5` и confirmation `top-3 × 25` это примерно:

- main Optuna: `480` backtests на компьютер;
- TS confirmation: `150` дополнительных backtests на компьютер;
- всего: `630` backtests на компьютер, `1260` backtests на оба запуска.

Если 1 backtest ≈ 1 минута, параллельный запуск на двух компьютерах займёт примерно **10.5 часов wall-clock**.


In [1]:
import json
import sys
import math
import gc
from pathlib import Path
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd

try:
    import optuna
except ImportError as exc:
    raise ImportError("Optuna не установлен. Установи: pip install optuna") from exc

PROJECT_ROOT = Path(r"D:\PythonProjects\VKR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.functions.klines_dataloader import KlinesDataLoader
from data_processing.functions.stream_indicators import stream_TA_lib
from data_processing.functions.transform_indicators import transform_indicators_df
from data_processing.functions.rolling_z_score_clip import rolling_z_score_clip_df
from backtest.backtest_class import Backtesting

print("Optuna version:", optuna.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)

Optuna version: 4.8.0
PROJECT_ROOT: D:\PythonProjects\VKR


D:\PythonProjects\VKR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Основные настройки эксперимента

Меняй только блок `RUN_PRESET` / `ALGORITHMS_TO_RUN`, если запускаешь notebook на разных компьютерах.

In [2]:
# =====================================================================
# Machine split
# =====================================================================
MACHINE_PRESETS = {
    # Recommended split: one TS + one UCB on each computer.
    "A_discounted": ["discounted_lints", "discounted_linucb"],
    "B_sliding": ["sw_lints", "sw_linucb"],
    "ALL": ["discounted_lints", "discounted_linucb", "sw_lints", "sw_linucb"],
}

RUN_PRESET = "A_discounted"  # <-- на втором компьютере поставь "B_sliding"
ALGORITHMS_TO_RUN = MACHINE_PRESETS[RUN_PRESET]

# Optional suffix to avoid overwriting outputs from different machines/runs.
RUN_LABEL = f"{RUN_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# =====================================================================
# Core protocol
# =====================================================================
BASE_SEED = 142
OPTUNA_SAMPLER_SEED = 20260515
ALL_SYMBOLS = ["BTCUSDT", "DOGEUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]
ACTIONS = [0, 1]

INTERVAL = 240
HORIZON = 10
TRADE_COST = 0.0025
OHLCV_RELATIVE_PATH = Path("data/klines_data/crypto_240m_bybit_TEST.parquet")

VAL_BARS = 2000
TEST_BARS = 2000
EMBARGO_BARS = 0

STATE_FEATURES = [
    "state_in_position",
    "state_log_bars_in_position",
    "state_unrealized_pnl",
]
META_COLS = ["timestamp", "symbol", "open", "high", "low", "close", "volume"]

# =====================================================================
# Feature-selection artifacts from screening stage
# =====================================================================
FEATURE_SELECTION_ROOT = PROJECT_ROOT / "feature_selection" / "hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2"
SCREENING_DIR = FEATURE_SELECTION_ROOT / "algorithm_specific_screening_defaults_ts20"
SELECTED_FOR_OPTUNA_JSON = SCREENING_DIR / "selected_feature_sets_for_optuna_balanced.json"

# No mixed set will be loaded. We use only selected corr/hybrid candidates.

# =====================================================================
# Local outputs; no database is used.
# =====================================================================
OUTPUT_ROOT = FEATURE_SELECTION_ROOT / "stage1_optuna_constrained_median_with_ts_confirmation_local"
OUTPUT_DIR = OUTPUT_ROOT / RUN_LABEL
DIAGNOSTICS_DIR = OUTPUT_DIR / "diagnostics"
STUDIES_DIR = OUTPUT_DIR / "studies"
FEATURE_CACHE_DIR = FEATURE_SELECTION_ROOT / "stage1_optuna_feature_cache"

for d in [OUTPUT_DIR, DIAGNOSTICS_DIR, STUDIES_DIR, FEATURE_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Fixed execution settings during Optuna.
# These are not tuned in Stage 1.
# =====================================================================
START_CAPITAL = 100.0
POSITION_SIZE = 0.10
MIN_HOLD_BARS = 4
COOLDOWN_BARS = 2
CONFIDENCE_THRESHOLD = 0.005
UPDATE_ON_VALIDATION = True

# =====================================================================
# Fixed reward clipping; not optimized.
# =====================================================================
REWARD_CLIP = 0.10

# =====================================================================
# Optuna budget.
# Recommended for thesis stage: 40 trials per algorithm-feature pair.
# TS is stochastic, so each trial is evaluated by median across 5 fixed seeds.
# UCB is deterministic enough; 1 seed is used.
# =====================================================================
N_TRIALS = 40
TS_SEEDS_PER_TRIAL = [3142, 3143, 3144, 3145, 3146]
UCB_SEEDS_PER_TRIAL = [3142]

# =====================================================================
# Constrained objective settings.
# Objective for valid trials:
#   median across seeds of median across symbols validation return_pct.
# Invalid trials get INVALID_SCORE and are not competitive in Optuna.
# =====================================================================
INVALID_SCORE = -1_000_000.0
MIN_TOTAL_TRADES = 250
REQUIRE_ALL_SEEDS_VALID = True

# =====================================================================
# Thompson Sampling confirmation stage.
# After main Optuna, top-3 TS trials per study are re-evaluated on 25 fresh seeds.
# This does not search new parameters; it only checks seed robustness of top trials.
# =====================================================================
RUN_TS_CONFIRMATION = True
TOP_N_CONFIRMATION = 3
CONFIRMATION_SEEDS = list(range(8000, 8025))

# Debug switches.
SMOKE_TEST = False
if SMOKE_TEST:
    N_TRIALS = 3
    TS_SEEDS_PER_TRIAL = [3142, 3143]
    UCB_SEEDS_PER_TRIAL = [3142]
    TOP_N_CONFIRMATION = 1
    CONFIRMATION_SEEDS = [8000, 8001]

# =====================================================================
# Indicator config must match feature-selection/screening pipeline.
# =====================================================================
CONFIG_FOR_INDICATORS = {
    "ema_periods": [9, 21, 50, 100, 200],
    "momentum_indicators_periods": [14, 30, 72],
    "return_indicators_periods": [6, 24, 72, 168],
    "volatility_indicators_periods": [24, 72, 168],
    "level_periods": [24, 72, 168],
    "vol_ma_period": [24, 72, 168],
    "range_ma_period": [24, 72, 168],
}

print("RUN_LABEL:", RUN_LABEL)
print("ALGORITHMS_TO_RUN:", ALGORITHMS_TO_RUN)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SELECTED_FOR_OPTUNA_JSON:", SELECTED_FOR_OPTUNA_JSON)
print("MIN_TOTAL_TRADES:", MIN_TOTAL_TRADES)
print("RUN_TS_CONFIRMATION:", RUN_TS_CONFIRMATION)
print("TOP_N_CONFIRMATION:", TOP_N_CONFIRMATION)
print("N_CONFIRMATION_SEEDS:", len(CONFIRMATION_SEEDS))


RUN_LABEL: A_discounted_20260515_234647
ALGORITHMS_TO_RUN: ['discounted_lints', 'discounted_linucb']
OUTPUT_DIR: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2\stage1_optuna_constrained_median_with_ts_confirmation_local\A_discounted_20260515_234647
SELECTED_FOR_OPTUNA_JSON: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2\algorithm_specific_screening_defaults_ts20\selected_feature_sets_for_optuna_balanced.json
MIN_TOTAL_TRADES: 250
RUN_TS_CONFIRMATION: True
TOP_N_CONFIRMATION: 3
N_CONFIRMATION_SEEDS: 25


## 2. Диапазоны Optuna

Финальный search space откалиброван вокруг screening defaults:

- `one_minus_gamma ∈ [0.001, 0.003]` → `gamma ∈ [0.997, 0.999]`, memory horizon примерно 333–1000 баров;
- `window_size ∈ [350, 1000]` с шагом 50;
- `lambda_prior ∈ [0.3, 10]` в log-scale;
- `noise_std ∈ [0.01, 0.08]` в log-scale для TS;
- `ucb_alpha ∈ [0.02, 0.30]` в log-scale для UCB.

`reward_clip` и execution-правила фиксированы: на Stage 1 оптимизируется только bandit dynamics, а не торговые правила исполнения.


In [3]:
# Search spaces.
# You can edit here, but keep ranges centered around screening defaults.
SEARCH_SPACE = {
    "discounted": {
        # gamma = 1 - one_minus_gamma => gamma in [0.997, 0.999]
        # memory horizon 1/(1-gamma) in ~[333, 1000] bars.
        "one_minus_gamma_low": 0.001,
        "one_minus_gamma_high": 0.003,
        "lambda_prior_low": 0.3,
        "lambda_prior_high": 10.0,
    },
    "sliding": {
        # Default W=500. Step=50 gives enough resolution without too many discrete values.
        "window_size_low": 350,
        "window_size_high": 1000,
        "window_size_step": 50,
        "lambda_prior_low": 0.3,
        "lambda_prior_high": 10.0,
    },
    "ts": {
        "noise_std_low": 0.01,
        "noise_std_high": 0.08,
    },
    "ucb": {
        "ucb_alpha_low": 0.02,
        "ucb_alpha_high": 0.30,
    },
}

TS_BANDITS = {"discounted_lints", "sw_lints"}
UCB_BANDITS = {"discounted_linucb", "sw_linucb"}
DISCOUNTED_BANDITS = {"discounted_lints", "discounted_linucb"}
SLIDING_BANDITS = {"sw_lints", "sw_linucb"}

BANDIT_TYPE_MAP = {
    "discounted_lints": "discounted_lints",
    "discounted_linucb": "discounted_linucb",
    "sw_lints": "sw_lints",
    "sw_linucb": "sw_linucb",
}

EXPECTED_MAIN_BACKTESTS_THIS_MACHINE = 0
EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE = 0
for algo in ALGORITHMS_TO_RUN:
    seeds = TS_SEEDS_PER_TRIAL if algo in TS_BANDITS else UCB_SEEDS_PER_TRIAL
    EXPECTED_MAIN_BACKTESTS_THIS_MACHINE += 2 * N_TRIALS * len(seeds)  # 2 feature sets per algorithm
    if RUN_TS_CONFIRMATION and algo in TS_BANDITS:
        EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE += 2 * TOP_N_CONFIRMATION * len(CONFIRMATION_SEEDS)

EXPECTED_MAIN_BACKTESTS_FULL = (
    2 * 2 * N_TRIALS * len(TS_SEEDS_PER_TRIAL)
    + 2 * 2 * N_TRIALS * len(UCB_SEEDS_PER_TRIAL)
)
EXPECTED_CONFIRMATION_BACKTESTS_FULL = (
    2 * 2 * TOP_N_CONFIRMATION * len(CONFIRMATION_SEEDS)
    if RUN_TS_CONFIRMATION else 0
)

print("N_TRIALS per study:", N_TRIALS)
print("Expected main Optuna backtests on this machine:", EXPECTED_MAIN_BACKTESTS_THIS_MACHINE)
print("Expected TS confirmation backtests on this machine:", EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE)
print("Expected total backtests on this machine:", EXPECTED_MAIN_BACKTESTS_THIS_MACHINE + EXPECTED_CONFIRMATION_BACKTESTS_THIS_MACHINE)
print("Expected main Optuna backtests full 4-algo run:", EXPECTED_MAIN_BACKTESTS_FULL)
print("Expected TS confirmation backtests full 4-algo run:", EXPECTED_CONFIRMATION_BACKTESTS_FULL)
print("Expected total backtests full 4-algo run:", EXPECTED_MAIN_BACKTESTS_FULL + EXPECTED_CONFIRMATION_BACKTESTS_FULL)


N_TRIALS per study: 40
Expected main Optuna backtests on this machine: 480
Expected TS confirmation backtests on this machine: 150
Expected total backtests on this machine: 630
Expected main Optuna backtests full 4-algo run: 960
Expected TS confirmation backtests full 4-algo run: 300
Expected total backtests full 4-algo run: 1260


## 3. Загрузка 8 candidates после screening

Файл `selected_feature_sets_for_optuna_balanced.json` должен содержать 2 набора для каждого алгоритма: лучший `corr` и лучший `hybrid`. Notebook дополнительно проверяет, что `mixed` set не попал в Optuna.

In [4]:
if not SELECTED_FOR_OPTUNA_JSON.exists():
    raise FileNotFoundError(f"Не найден файл selected feature sets: {SELECTED_FOR_OPTUNA_JSON}")

with open(SELECTED_FOR_OPTUNA_JSON, "r", encoding="utf-8") as f:
    SELECTED_FOR_OPTUNA = json.load(f)

# Keep only requested algorithms.
missing_algos = [a for a in ALGORITHMS_TO_RUN if a not in SELECTED_FOR_OPTUNA]
if missing_algos:
    raise ValueError(f"В selected JSON нет алгоритмов: {missing_algos}")

FEATURE_PAIRS = []
for algo in ALGORITHMS_TO_RUN:
    items = SELECTED_FOR_OPTUNA[algo]
    if len(items) != 2:
        raise ValueError(f"Ожидалось 2 candidates для {algo}, получено {len(items)}")
    for item in items:
        method_group = item["method_group"]
        set_name = item["set_name"]
        if method_group not in {"corr", "hybrid"}:
            raise ValueError(f"Mixed/unknown method group is not allowed in this notebook: {method_group}")
        if "mixed" in set_name.lower():
            raise ValueError(f"Mixed set is not allowed in this notebook: {set_name}")
        FEATURES = list(item["features"])
        META = dict(item["meta"])
        FEATURE_PAIRS.append({
            "algorithm": algo,
            "method_group": method_group,
            "set_name": set_name,
            "features": FEATURES,
            "meta": META,
            "z_window": int(META["z_window"]),
            "alpha_out": float(META["alpha_out"]),
        })

feature_pairs_table = pd.DataFrame([
    {
        "algorithm": p["algorithm"],
        "method_group": p["method_group"],
        "set_name": p["set_name"],
        "z_window": p["z_window"],
        "n_features": len(p["features"]),
        "features": "|".join(p["features"]),
    }
    for p in FEATURE_PAIRS
])

feature_pairs_table.to_csv(OUTPUT_DIR / "feature_pairs_to_optimize.csv", index=False)
display(feature_pairs_table)

needed_z_windows = sorted(set(feature_pairs_table["z_window"].astype(int)))
print("needed_z_windows:", needed_z_windows)

,algorithm,method_group,set_name,z_window,n_features,features
0,discounted_lints,corr,z24_a0p5_corr_pruned_top10,24,10,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
1,discounted_lints,hybrid,z24_a0p5_hybrid_corr5_stablehie5_top10,24,10,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
2,discounted_linucb,corr,z72_a0p5_corr_pruned_top10,72,10,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
3,discounted_linucb,hybrid,z24_a0p5_hybrid_corr5_stablehie5_top10,24,10,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...


needed_z_windows: [24, 72]


## 4. Data processing helpers

In [5]:
def process_indicators_for_z_window(
    ohlcv_data: pd.DataFrame,
    symbols: list[str],
    meta_cols: list[str],
    indicator_config: dict,
    z_window: int,
    clip_value: float = 5.0,
    shift_by_one: bool = True,
) -> pd.DataFrame:
    processed_parts = []
    for symbol in symbols:
        print(f"Processing {symbol}, z_window={z_window}...")
        df_symbol = (
            ohlcv_data[ohlcv_data["symbol"] == symbol]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        indicators_symbol = stream_TA_lib(
            df=df_symbol,
            meta_cols=meta_cols,
            **indicator_config,
        )
        indicators_symbol = indicators_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        transformed_symbol = transform_indicators_df(
            df=indicators_symbol,
            meta_cols=meta_cols,
        )
        z_score_symbol = rolling_z_score_clip_df(
            df=transformed_symbol,
            meta_cols=meta_cols,
            window=z_window,
            clip_value=clip_value,
            shift_by_one=shift_by_one,
        )
        z_score_symbol = z_score_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        processed_parts.append(z_score_symbol)

    return (
        pd.concat(processed_parts, ignore_index=True)
        .sort_values(["symbol", "timestamp"])
        .reset_index(drop=True)
    )


def split_train_val_test_by_tail_bars(
    df: pd.DataFrame,
    symbols: list[str],
    val_bars: int,
    test_bars: int,
    embargo_bars: int = 0,
):
    train_parts, val_parts, test_parts, rows = [], [], [], []
    for sym in symbols:
        g = df[df["symbol"] == sym].sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)
        if n <= val_bars + test_bars + max(embargo_bars, 0):
            raise ValueError(f"Too few rows for {sym}: {n}")

        test_start = n - test_bars
        val_start = test_start - val_bars
        train_end = max(val_start - embargo_bars, 0)
        val_end = test_start - embargo_bars if embargo_bars > 0 else test_start

        train = g.iloc[:train_end].copy()
        val = g.iloc[val_start:val_end].copy()
        test = g.iloc[test_start:].copy()

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)
        rows.append({
            "symbol": sym,
            "n_total": n,
            "n_train": len(train),
            "n_val": len(val),
            "n_test": len(test),
            "train_min": train["timestamp"].min(),
            "train_max": train["timestamp"].max(),
            "val_min": val["timestamp"].min(),
            "val_max": val["timestamp"].max(),
            "test_min": test["timestamp"].min(),
            "test_max": test["timestamp"].max(),
        })

    split_summary = pd.DataFrame(rows)
    train_df = pd.concat(train_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    test_df = pd.concat(test_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return train_df, val_df, test_df, split_summary


def validate_selected_features(train_df: pd.DataFrame, val_df: pd.DataFrame, selected_features: list[str], set_name: str):
    missing_train = [f for f in selected_features if f not in train_df.columns]
    missing_val = [f for f in selected_features if f not in val_df.columns]
    if missing_train or missing_val:
        raise ValueError(f"Missing selected features for {set_name}: train={missing_train}, val={missing_val}")
    values_train = train_df[selected_features].to_numpy(dtype=float)
    values_val = val_df[selected_features].to_numpy(dtype=float)
    if np.isnan(values_train).any() or np.isnan(values_val).any():
        raise ValueError(f"NaN in selected features for {set_name}")
    if np.isinf(values_train).any() or np.isinf(values_val).any():
        raise ValueError(f"inf in selected features for {set_name}")

## 5. Build datasets by z-window

Используется cache в отдельной папке `stage1_optuna_feature_cache`, чтобы не пересчитывать признаки при повторном запуске.

In [6]:
loader = KlinesDataLoader(symbols=ALL_SYMBOLS)
ohlcv_data = loader.load_data(
    download_path=str(OHLCV_RELATIVE_PATH).replace("\\", "/"),
    analyse_data=False,
    cleaning=True,
)
ohlcv_data["timestamp"] = pd.to_datetime(ohlcv_data["timestamp"], utc=True)

DATASETS_BY_Z: dict[int, dict[str, Any]] = {}
split_summaries = []

for z_window in needed_z_windows:
    cache_path = FEATURE_CACHE_DIR / f"datasets_z{z_window}_val{VAL_BARS}_test{TEST_BARS}_embargo{EMBARGO_BARS}.pkl"
    if cache_path.exists():
        print(f"Loading cached dataset for z_window={z_window}: {cache_path}")
        cached = pd.read_pickle(cache_path)
        DATASETS_BY_Z[z_window] = cached["datasets"]
        split_summary = cached["split_summary"].copy()
    else:
        print("=" * 100)
        print(f"Building z_window={z_window}")
        print("=" * 100)
        processed = process_indicators_for_z_window(
            ohlcv_data=ohlcv_data,
            symbols=ALL_SYMBOLS,
            meta_cols=META_COLS,
            indicator_config=CONFIG_FOR_INDICATORS,
            z_window=z_window,
            clip_value=5.0,
            shift_by_one=True,
        )
        train_z, val_z, test_z, split_summary = split_train_val_test_by_tail_bars(
            df=processed,
            symbols=ALL_SYMBOLS,
            val_bars=VAL_BARS,
            test_bars=TEST_BARS,
            embargo_bars=EMBARGO_BARS,
        )
        DATASETS_BY_Z[z_window] = {"train": train_z, "val": val_z, "test": test_z}
        pd.to_pickle({"datasets": DATASETS_BY_Z[z_window], "split_summary": split_summary}, cache_path)

    split_summary = split_summary.copy()
    split_summary.insert(0, "z_window", z_window)
    split_summaries.append(split_summary)

split_summary_used = pd.concat(split_summaries, ignore_index=True)
split_summary_used.to_csv(DIAGNOSTICS_DIR / "split_summary_used.csv", index=False)
display(split_summary_used)

# Validate selected features.
for p in FEATURE_PAIRS:
    z = p["z_window"]
    validate_selected_features(
        train_df=DATASETS_BY_Z[z]["train"],
        val_df=DATASETS_BY_Z[z]["val"],
        selected_features=p["features"],
        set_name=p["set_name"],
    )
print("All selected feature sets validated.")

Building z_window=24
Processing BTCUSDT, z_window=24...
Processing DOGEUSDT, z_window=24...
Processing ETHUSDT, z_window=24...
Processing SOLUSDT, z_window=24...
Processing XRPUSDT, z_window=24...
Building z_window=72
Processing BTCUSDT, z_window=72...
Processing DOGEUSDT, z_window=72...
Processing ETHUSDT, z_window=72...
Processing SOLUSDT, z_window=72...
Processing XRPUSDT, z_window=72...


,z_window,symbol,n_total,n_train,n_val,n_test,train_min,train_max,val_min,val_max,test_min,test_max
0,24,BTCUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
1,24,DOGEUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
2,24,ETHUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
3,24,SOLUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
4,24,XRPUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
5,72,BTCUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
6,72,DOGEUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
7,72,ETHUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
8,72,SOLUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
9,72,XRPUSDT,9626,5626,2000,2000,2021-12-06 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00


All selected feature sets validated.


## 6. Backtest metrics helpers

In [7]:
def _get_store(backtesting, mode: str, store_name: str):
    if mode == "train":
        return getattr(backtesting, f"{store_name}_train")
    if mode == "val":
        return getattr(backtesting, f"{store_name}_val")
    raise ValueError("mode должен быть 'train' или 'val'")


def extract_trade_stats_from_log(trade_log: list[dict], trade_cost: float) -> pd.DataFrame:
    if not trade_log:
        return pd.DataFrame()
    events = pd.DataFrame(trade_log).copy()
    if events.empty or "event" not in events.columns:
        return pd.DataFrame()
    trades = []
    open_trade = None
    for _, row in events.sort_values("timestamp").iterrows():
        if row["event"] == "BUY":
            open_trade = row
        elif row["event"] == "SELL" and open_trade is not None:
            entry_price = float(open_trade["price"])
            exit_price = float(row["price"])
            pnl_before_cost = np.log(exit_price / entry_price)
            pnl_after_cost = pnl_before_cost + 2.0 * np.log(1.0 - trade_cost)
            trades.append({
                "entry_timestamp": open_trade["timestamp"],
                "exit_timestamp": row["timestamp"],
                "entry_price": entry_price,
                "exit_price": exit_price,
                "pnl_before_cost": pnl_before_cost,
                "pnl_after_cost": pnl_after_cost,
                "holding_time": pd.to_datetime(row["timestamp"]) - pd.to_datetime(open_trade["timestamp"]),
            })
            open_trade = None
    return pd.DataFrame(trades)


def compute_symbol_backtest_metrics(backtesting, sym: str, mode: str, trade_cost: float) -> dict:
    balance_store = _get_store(backtesting, mode, "balance")
    actions_store = _get_store(backtesting, mode, "actions")
    raw_actions_store = _get_store(backtesting, mode, "raw_actions")
    decision_log_store = _get_store(backtesting, mode, "decision_log")
    trade_log_store = _get_store(backtesting, mode, "trade_log")
    reward_log_store = _get_store(backtesting, mode, "reward_log")

    balance = np.asarray(balance_store.get(sym, []), dtype=float)
    actions = np.asarray(actions_store.get(sym, []), dtype=float)
    raw_actions = np.asarray(raw_actions_store.get(sym, []), dtype=float)
    decision_log = pd.DataFrame(decision_log_store.get(sym, []))
    trade_log = trade_log_store.get(sym, [])
    reward_log = pd.DataFrame(reward_log_store.get(sym, []))

    if len(balance) == 0:
        return {"symbol": sym, "mode": mode, "n_decisions": 0}

    start_value = float(balance[0])
    end_value = float(balance[-1])
    return_pct = (end_value / start_value - 1.0) * 100.0
    running_max = np.maximum.accumulate(balance)
    drawdown = balance / (running_max + 1e-12) - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    rows = {
        "symbol": sym,
        "mode": mode,
        "start_value": start_value,
        "end_value": end_value,
        "return_pct": return_pct,
        "max_drawdown_pct": max_drawdown_pct,
        "drawdown_abs": abs(max_drawdown_pct),
        "raw_action_1_ratio": float(np.mean(raw_actions == 1)) if len(raw_actions) else np.nan,
        "executed_action_1_ratio": float(np.mean(actions == 1)) if len(actions) else np.nan,
        "executed_action_1": int(np.sum(actions == 1)) if len(actions) else 0,
        "raw_action_1": int(np.sum(raw_actions == 1)) if len(raw_actions) else 0,
        "n_decisions": int(len(actions)),
    }

    if not decision_log.empty:
        rows.update({
            "threshold_applied_ratio": float(decision_log["threshold_applied"].mean()),
            "constraint_applied_ratio": float(decision_log["constraint_applied"].mean()),
            "abs_edge_mean": float(decision_log["abs_edge"].mean()),
            "abs_edge_p50": float(decision_log["abs_edge"].quantile(0.50)),
            "uncertainty_0_mean": float(decision_log["uncertainty_0"].mean()),
            "uncertainty_1_mean": float(decision_log["uncertainty_1"].mean()),
        })
    else:
        rows.update({
            "threshold_applied_ratio": np.nan,
            "constraint_applied_ratio": np.nan,
            "abs_edge_mean": np.nan,
            "abs_edge_p50": np.nan,
            "uncertainty_0_mean": np.nan,
            "uncertainty_1_mean": np.nan,
        })

    if not reward_log.empty and "reward" in reward_log.columns:
        rew = reward_log["reward"].astype(float)
        rows.update({
            "n_reward_updates": int(len(reward_log)),
            "mean_reward_update": float(rew.mean()),
            "std_reward_update": float(rew.std()),
            "reward_abs_p95": float(rew.abs().quantile(0.95)),
        })
    else:
        rows.update({
            "n_reward_updates": 0,
            "mean_reward_update": np.nan,
            "std_reward_update": np.nan,
            "reward_abs_p95": np.nan,
        })

    trades = extract_trade_stats_from_log(trade_log=trade_log, trade_cost=trade_cost)
    if not trades.empty:
        gross_profit = trades.loc[trades["pnl_after_cost"] > 0, "pnl_after_cost"].sum()
        gross_loss = trades.loc[trades["pnl_after_cost"] < 0, "pnl_after_cost"].sum()
        rows.update({
            "trades": int(len(trades)),
            "win_rate": float((trades["pnl_after_cost"] > 0).mean()),
            "mean_trade_pnl": float(trades["pnl_after_cost"].mean()),
            "median_trade_pnl": float(trades["pnl_after_cost"].median()),
            "total_trade_pnl": float(trades["pnl_after_cost"].sum()),
            "profit_factor": float(gross_profit / abs(gross_loss + 1e-12)),
        })
    else:
        rows.update({
            "trades": 0,
            "win_rate": np.nan,
            "mean_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
            "total_trade_pnl": 0.0,
            "profit_factor": np.nan,
        })
    return rows


def summarize_seed_validation(symbol_metrics: pd.DataFrame) -> dict:
    val = symbol_metrics[symbol_metrics["mode"].eq("val")].copy()
    if val.empty:
        raise ValueError("No validation metrics")
    val["profit_factor_clean"] = val["profit_factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    val["is_active"] = val["trades"] > 0
    val["has_long_actions"] = val["executed_action_1"] > 0
    val["is_positive_return"] = val["return_pct"] > 0
    return {
        "n_symbols": int(val["symbol"].nunique()),
        "mean_return_pct": float(val["return_pct"].mean()),
        "median_return_pct": float(val["return_pct"].median()),
        "min_return_pct": float(val["return_pct"].min()),
        "max_return_pct": float(val["return_pct"].max()),
        "max_drawdown_abs": float(val["drawdown_abs"].max()),
        "mean_drawdown_abs": float(val["drawdown_abs"].mean()),
        "median_profit_factor": float(val["profit_factor_clean"].median()),
        "mean_profit_factor": float(val["profit_factor_clean"].mean()),
        "total_trades": int(val["trades"].sum()),
        "mean_trades": float(val["trades"].mean()),
        "mean_win_rate": float(val["win_rate"].mean()),
        "mean_raw_action_1_ratio": float(val["raw_action_1_ratio"].mean()),
        "mean_executed_action_1_ratio": float(val["executed_action_1_ratio"].mean()),
        "mean_threshold_applied_ratio": float(val["threshold_applied_ratio"].mean()),
        "mean_constraint_applied_ratio": float(val["constraint_applied_ratio"].mean()),
        "active_symbols": int(val["is_active"].sum()),
        "long_action_symbols": int(val["has_long_actions"].sum()),
        "positive_return_symbols": int(val["is_positive_return"].sum()),
        "all_symbols_active": bool(val["is_active"].sum() == val["symbol"].nunique()),
        "all_symbols_have_long": bool(val["has_long_actions"].sum() == val["symbol"].nunique()),
    }


def aggregate_seed_summaries(seed_summaries: list[dict]) -> dict:
    s = pd.DataFrame(seed_summaries)
    return {
        "n_seeds": int(s["seed"].nunique()),
        "seeds_used": "|".join(str(int(v)) for v in sorted(s["seed"].unique())),
        "all_symbols_active_share": float(s["all_symbols_active"].mean()),
        "all_symbols_have_long_share": float(s["all_symbols_have_long"].mean()),
        "median_of_seed_median_return_pct": float(s["median_return_pct"].median()),
        "mean_of_seed_median_return_pct": float(s["median_return_pct"].mean()),
        "std_of_seed_median_return_pct": float(s["median_return_pct"].std(ddof=1)) if len(s) > 1 else 0.0,
        "min_seed_median_return_pct": float(s["median_return_pct"].min()),
        "max_seed_median_return_pct": float(s["median_return_pct"].max()),
        "median_of_seed_mean_return_pct": float(s["mean_return_pct"].median()),
        "median_max_drawdown_abs": float(s["max_drawdown_abs"].median()),
        "median_profit_factor": float(s["median_profit_factor"].median()),
        "median_total_trades": float(s["total_trades"].median()),
        "median_positive_return_symbols": float(s["positive_return_symbols"].median()),
        "median_active_symbols": float(s["active_symbols"].median()),
        "median_long_action_symbols": float(s["long_action_symbols"].median()),
    }

## 7. Constrained objective: median-of-medians

Optuna maximizes only one scalar metric without subjective weights:

\[
\text{objective} = median_{seed}(median_{symbol}(return\_pct))
\]

Hard constraints are applied before the objective:

- all symbols must be active;
- all symbols must have long actions;
- median total trades must be at least `MIN_TOTAL_TRADES = 250`;
- constraints must hold for all TS seeds if `REQUIRE_ALL_SEEDS_VALID=True`.

Invalid trials receive `INVALID_SCORE = -1_000_000`.

Risk metrics (`drawdown`, `profit_factor`, `trades`) are saved and used only for final lexicographic ranking / interpretation, not as weighted terms inside the Optuna objective.


In [8]:
def suggest_bandit_params(trial: optuna.Trial, algorithm: str) -> dict:
    params = {
        "bandit_type": BANDIT_TYPE_MAP[algorithm],
        "lambda_prior": trial.suggest_float(
            "lambda_prior",
            SEARCH_SPACE["discounted" if algorithm in DISCOUNTED_BANDITS else "sliding"]["lambda_prior_low"],
            SEARCH_SPACE["discounted" if algorithm in DISCOUNTED_BANDITS else "sliding"]["lambda_prior_high"],
            log=True,
        ),
        "reward_clip": REWARD_CLIP,
    }

    if algorithm in DISCOUNTED_BANDITS:
        one_minus_gamma = trial.suggest_float(
            "one_minus_gamma",
            SEARCH_SPACE["discounted"]["one_minus_gamma_low"],
            SEARCH_SPACE["discounted"]["one_minus_gamma_high"],
            log=True,
        )
        params["discount_factor"] = 1.0 - one_minus_gamma
        trial.set_user_attr("memory_horizon", 1.0 / one_minus_gamma)
    else:
        params["window_size"] = trial.suggest_int(
            "window_size",
            SEARCH_SPACE["sliding"]["window_size_low"],
            SEARCH_SPACE["sliding"]["window_size_high"],
            step=SEARCH_SPACE["sliding"]["window_size_step"],
        )

    if algorithm in TS_BANDITS:
        params["noise_std"] = trial.suggest_float(
            "noise_std",
            SEARCH_SPACE["ts"]["noise_std_low"],
            SEARCH_SPACE["ts"]["noise_std_high"],
            log=True,
        )
    else:
        params["ucb_alpha"] = trial.suggest_float(
            "ucb_alpha",
            SEARCH_SPACE["ucb"]["ucb_alpha_low"],
            SEARCH_SPACE["ucb"]["ucb_alpha_high"],
            log=True,
        )
    return params


def default_trial_params(algorithm: str) -> dict:
    """Known-good screening defaults forced as first Optuna trial via enqueue_trial."""
    if algorithm in DISCOUNTED_BANDITS:
        params = {
            "one_minus_gamma": 0.002,  # gamma=0.998
            "lambda_prior": 1.0,
        }
    else:
        params = {
            "window_size": 500,
            "lambda_prior": 1.0,
        }

    if algorithm in TS_BANDITS:
        params["noise_std"] = 0.03
    else:
        params["ucb_alpha"] = 0.10

    return params


def constrained_median_objective_from_agg(agg: dict) -> tuple[float, dict]:
    """
    Hard-constrained objective.

    Valid trial objective:
        median_of_seed_median_return_pct.

    Invalid trial objective:
        INVALID_SCORE.
    """
    if REQUIRE_ALL_SEEDS_VALID:
        symbols_active_ok = float(agg["all_symbols_active_share"]) >= 1.0
        symbols_have_long_ok = float(agg["all_symbols_have_long_share"]) >= 1.0
    else:
        # Optional softer mode: all constraints must hold for most seeds.
        symbols_active_ok = float(agg["all_symbols_active_share"]) >= 0.80
        symbols_have_long_ok = float(agg["all_symbols_have_long_share"]) >= 0.80

    trades_ok = float(agg["median_total_trades"]) >= float(MIN_TOTAL_TRADES)
    finite_return_ok = np.isfinite(float(agg["median_of_seed_median_return_pct"]))

    valid = bool(symbols_active_ok and symbols_have_long_ok and trades_ok and finite_return_ok)

    if valid:
        score = float(agg["median_of_seed_median_return_pct"])
    else:
        score = float(INVALID_SCORE)

    details = {
        "valid": valid,
        "constraint_symbols_active_ok": bool(symbols_active_ok),
        "constraint_symbols_have_long_ok": bool(symbols_have_long_ok),
        "constraint_trades_ok": bool(trades_ok),
        "constraint_finite_return_ok": bool(finite_return_ok),
        "objective_metric_name": "median_of_seed_median_return_pct",
        "objective_score": score,
        "invalid_score": float(INVALID_SCORE),
        "min_total_trades": float(MIN_TOTAL_TRADES),
        "require_all_seeds_valid": bool(REQUIRE_ALL_SEEDS_VALID),
    }
    return score, details


def run_backtest_for_seed(
    algorithm: str,
    feature_pair: dict,
    bandit_params: dict,
    seed: int,
) -> tuple[pd.DataFrame, dict]:
    z_window = int(feature_pair["z_window"])
    selected_features = list(feature_pair["features"])
    train_z = DATASETS_BY_Z[z_window]["train"]
    val_z = DATASETS_BY_Z[z_window]["val"]

    config_for_bandit = dict(bandit_params)
    config_for_bandit["n_features"] = len(selected_features) + len(STATE_FEATURES)
    config_for_bandit["actions"] = ACTIONS
    config_for_bandit["seed"] = int(seed)

    bt = Backtesting(
        meta_cols=META_COLS,
        feature_columns=selected_features,
        config_for_bandit=config_for_bandit,
        trade_cost=TRADE_COST,
        seed=int(seed),
        update_on_validation=UPDATE_ON_VALIDATION,
        horizon=HORIZON,
        min_hold_bars=MIN_HOLD_BARS,
        cooldown_bars=COOLDOWN_BARS,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        alpha_out=float(feature_pair["alpha_out"]),
        state_feature_columns=STATE_FEATURES,
    )
    bt.backtest(
        dataframe_train=train_z,
        dataframe_val=val_z,
        symbols=ALL_SYMBOLS,
        start_capital=START_CAPITAL,
        position_size=POSITION_SIZE,
    )

    metric_rows = []
    for sym in ALL_SYMBOLS:
        for mode in ["train", "val"]:
            metric_rows.append(compute_symbol_backtest_metrics(bt, sym, mode, TRADE_COST))
    symbol_metrics = pd.DataFrame(metric_rows)
    seed_summary = summarize_seed_validation(symbol_metrics)
    seed_summary["seed"] = int(seed)
    return symbol_metrics, seed_summary


def append_jsonl(path: Path, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, default=str) + "\n")


## 8. Run Optuna studies

Notebook запускает отдельное study для каждой пары `(algorithm, feature set)`. Storage не указан, поэтому база данных не создаётся. После каждого trial строка результата дописывается в локальный `.jsonl` и `.csv`.

Первый trial каждого study форсируется через `study.enqueue_trial(default_trial_params(...))`, чтобы known-good screening defaults всегда присутствовали в Optuna-run.


In [ ]:
all_trial_rows = []
all_seed_rows = []
all_symbol_rows = []
study_best_rows = []
error_rows = []

# Save global config for reproducibility.
run_config = {
    "created_at": datetime.now().isoformat(),
    "run_label": RUN_LABEL,
    "algorithms_to_run": ALGORITHMS_TO_RUN,
    "n_trials": N_TRIALS,
    "ts_seeds_per_trial": TS_SEEDS_PER_TRIAL,
    "ucb_seeds_per_trial": UCB_SEEDS_PER_TRIAL,
    "search_space": SEARCH_SPACE,
    "reward_clip_fixed": REWARD_CLIP,
    "objective": {
        "type": "constrained_median_of_medians",
        "primary_metric": "median_of_seed_median_return_pct",
        "invalid_score": INVALID_SCORE,
        "min_total_trades": MIN_TOTAL_TRADES,
        "require_all_seeds_valid": REQUIRE_ALL_SEEDS_VALID,
        "no_subjective_weights": True,
        "ts_confirmation_stage": RUN_TS_CONFIRMATION,
        "top_n_confirmation": TOP_N_CONFIRMATION,
        "confirmation_seeds": CONFIRMATION_SEEDS,
    },
    "execution": {
        "start_capital": START_CAPITAL,
        "position_size": POSITION_SIZE,
        "min_hold_bars": MIN_HOLD_BARS,
        "cooldown_bars": COOLDOWN_BARS,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "update_on_validation": UPDATE_ON_VALIDATION,
        "horizon": HORIZON,
        "trade_cost": TRADE_COST,
    },
    "feature_pairs": feature_pairs_table.to_dict(orient="records"),
}
(OUTPUT_DIR / "optuna_stage1_config.json").write_text(json.dumps(run_config, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

trial_jsonl_path = OUTPUT_DIR / "trial_results.jsonl"
error_jsonl_path = OUTPUT_DIR / "trial_errors.jsonl"

for pair_idx, feature_pair in enumerate(FEATURE_PAIRS, start=1):
    algorithm = feature_pair["algorithm"]
    method_group = feature_pair["method_group"]
    set_name = feature_pair["set_name"]
    selected_features = feature_pair["features"]
    seeds_for_trial = TS_SEEDS_PER_TRIAL if algorithm in TS_BANDITS else UCB_SEEDS_PER_TRIAL

    study_name = f"{algorithm}__{method_group}__{set_name}"
    safe_study_name = study_name.replace("/", "_").replace("\\", "_").replace(":", "_")
    study_dir = STUDIES_DIR / safe_study_name
    study_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 120)
    print(f"Study {pair_idx}/{len(FEATURE_PAIRS)}: {study_name}")
    print("features:", selected_features)
    print("seeds:", seeds_for_trial)
    print("=" * 120)

    sampler = optuna.samplers.TPESampler(
        seed=OPTUNA_SAMPLER_SEED + pair_idx,
        n_startup_trials=min(10, max(1, N_TRIALS // 4)),
        multivariate=True,
        group=True,
    )
    study = optuna.create_study(
        study_name=study_name,
        direction="maximize",
        sampler=sampler,
        pruner=optuna.pruners.NopPruner(),
        storage=None,
    )

    # Force known-good screening defaults as the first trial in every study.
    # This keeps a default baseline inside Optuna regardless of TPE startup randomness.
    study.enqueue_trial(default_trial_params(algorithm))

    def objective(trial: optuna.Trial) -> float:
        bandit_params = suggest_bandit_params(trial, algorithm)
        seed_summaries = []
        symbol_metric_frames = []
        try:
            for seed in seeds_for_trial:
                symbol_metrics, seed_summary = run_backtest_for_seed(
                    algorithm=algorithm,
                    feature_pair=feature_pair,
                    bandit_params=bandit_params,
                    seed=int(seed),
                )
                symbol_metrics.insert(0, "seed", int(seed))
                symbol_metrics.insert(0, "trial_number", int(trial.number))
                symbol_metrics.insert(0, "set_name", set_name)
                symbol_metrics.insert(0, "method_group", method_group)
                symbol_metrics.insert(0, "bandit_name", algorithm)
                symbol_metric_frames.append(symbol_metrics)

                seed_summary.update({
                    "trial_number": int(trial.number),
                    "bandit_name": algorithm,
                    "method_group": method_group,
                    "set_name": set_name,
                    "z_window": feature_pair["z_window"],
                })
                seed_summaries.append(seed_summary)

            agg = aggregate_seed_summaries(seed_summaries)
            score, constraint_details = constrained_median_objective_from_agg(agg)

            trial_row = {
                "trial_number": int(trial.number),
                "study_name": study_name,
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": set_name,
                "z_window": feature_pair["z_window"],
                "alpha_out": feature_pair["alpha_out"],
                "n_market_features": len(selected_features),
                "feature_list": "|".join(selected_features),
                "objective_score": score,
                **constraint_details,
                **agg,
                **{f"param_{k}": v for k, v in bandit_params.items() if k not in {"actions"}},
                **{f"optuna_param_{k}": v for k, v in trial.params.items()},
            }

            all_trial_rows.append(trial_row)
            all_seed_rows.extend(seed_summaries)
            if symbol_metric_frames:
                all_symbol_rows.append(pd.concat(symbol_metric_frames, ignore_index=True))

            append_jsonl(trial_jsonl_path, trial_row)
            pd.DataFrame(all_trial_rows).to_csv(OUTPUT_DIR / "trial_results_all.csv", index=False)
            pd.DataFrame(all_seed_rows).to_csv(OUTPUT_DIR / "trial_seed_level_summary_all.csv", index=False)
            if all_symbol_rows:
                pd.concat(all_symbol_rows, ignore_index=True).to_csv(DIAGNOSTICS_DIR / "trial_symbol_metrics_all.csv", index=False)

            trial.set_user_attr("objective_score", score)
            for k, v in constraint_details.items():
                trial.set_user_attr(k, v)
            for k, v in agg.items():
                trial.set_user_attr(k, v)
            return score

        except Exception as e:
            err = {
                "trial_number": int(trial.number),
                "study_name": study_name,
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": set_name,
                "error": repr(e),
                "params": dict(trial.params),
            }
            error_rows.append(err)
            append_jsonl(error_jsonl_path, err)
            pd.DataFrame(error_rows).to_csv(OUTPUT_DIR / "trial_errors.csv", index=False)
            print("TRIAL ERROR:", err)
            return -1e12

    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, gc_after_trial=True, show_progress_bar=True)

    # Save Optuna's native trial table for this study.
    study_trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
    study_trials_df.to_csv(study_dir / "optuna_trials_dataframe.csv", index=False)

    best = study.best_trial
    best_row = {
        "study_name": study_name,
        "bandit_name": algorithm,
        "method_group": method_group,
        "set_name": set_name,
        "best_trial_number": int(best.number),
        "best_objective_score": float(best.value),
        **{f"best_param_{k}": v for k, v in best.params.items()},
        **{f"best_user_{k}": v for k, v in best.user_attrs.items()},
    }
    study_best_rows.append(best_row)
    pd.DataFrame(study_best_rows).to_csv(OUTPUT_DIR / "study_best_summary.csv", index=False)

    with open(study_dir / "best_trial.json", "w", encoding="utf-8") as f:
        json.dump(best_row, f, ensure_ascii=False, indent=2, default=str)

    gc.collect()

print("Done.")
print("OUTPUT_DIR:", OUTPUT_DIR)

C:\Users\trosh\AppData\Local\Temp\ipykernel_23648\1728386642.py:63: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
C:\Users\trosh\AppData\Local\Temp\ipykernel_23648\1728386642.py:63: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-05-15 23:46:51,567] A new study created in memory with name: discounted_lints__corr__z24_a0p5_corr_pruned_top10


Study 1/4: discounted_lints__corr__z24_a0p5_corr_pruned_top10
features: ['ret_accel_6_24_z', 'volatility_72_sqrt_z', 'vol_ratio_168_log1p_z', 'range_sqrt_z', 'MOM_pct_72_z', 'ret_accel_24_72_z', 'MOM_pct_30_z', 'ADX_14_scaled', 'NATR_14_z', 'volatility_168_sqrt_z']
seeds: [3142, 3143, 3144, 3145, 3146]


  0%|          | 0/40 [00:00<?, ?it/s]

BTCUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
DOGEUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
DOGEUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
DOGEUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
DOGEUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
ETHUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
ETHUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
ETHUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
ETHUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
SOLUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
SOLUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
SOLUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
SOLUSDT: фаза val закончилась: 2025-05-30 04:00:00+00:00
XRPUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
XRPUSDT: фаза train закончилась: 202

## 9. Final ranking for this machine

Сортировка после Optuna:

1. valid trials first;
2. higher constrained objective score;
3. higher median-of-median return;
4. higher mean-of-median return;
5. lower drawdown;
6. higher profit factor;
7. more trades.

Основной objective для Optuna — только `median_of_seed_median_return_pct` после hard constraints. Остальные метрики используются как tie-breakers для отчёта и анализа.


In [ ]:
trial_results = pd.read_csv(OUTPUT_DIR / "trial_results_all.csv")

ranking_cols = [
    "valid",
    "objective_score",
    "median_of_seed_median_return_pct",
    "mean_of_seed_median_return_pct",
    "median_max_drawdown_abs",
    "median_profit_factor",
    "median_total_trades",
]

final_ranking = (
    trial_results
    .sort_values(
        ["bandit_name", "method_group", *ranking_cols],
        ascending=[True, True, False, False, False, False, True, False, False],
    )
    .reset_index(drop=True)
)
final_ranking["rank_within_algorithm_method"] = final_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1

# Best trial for every algorithm × method group.
best_per_pair = final_ranking[final_ranking["rank_within_algorithm_method"].eq(1)].copy()

# Best method comparison within each algorithm.
best_method_comparison = (
    best_per_pair
    .sort_values(
        ["bandit_name", *ranking_cols],
        ascending=[True, False, False, False, False, True, False, False],
    )
    .reset_index(drop=True)
)
best_method_comparison["rank_within_algorithm"] = best_method_comparison.groupby("bandit_name").cumcount() + 1

final_ranking.to_csv(OUTPUT_DIR / "final_trial_ranking_all.csv", index=False)
best_per_pair.to_csv(OUTPUT_DIR / "best_trial_per_algorithm_method.csv", index=False)
best_method_comparison.to_csv(OUTPUT_DIR / "best_method_comparison_within_algorithm.csv", index=False)

display(best_per_pair[[
    "bandit_name", "method_group", "set_name", "trial_number", "valid",
    "median_of_seed_median_return_pct", "mean_of_seed_median_return_pct",
    "median_max_drawdown_abs", "median_profit_factor", "median_total_trades",
    "objective_score",
]])

display(best_method_comparison[[
    "bandit_name", "method_group", "set_name", "trial_number", "rank_within_algorithm", "valid",
    "median_of_seed_median_return_pct", "mean_of_seed_median_return_pct",
    "median_max_drawdown_abs", "median_profit_factor", "median_total_trades", "objective_score",
]])


## 10. TS confirmation: top-3 trials × 25 fresh seeds

Для Thompson Sampling (`discounted_lints`, `sw_lints`) после основного Optuna top-3 trials в каждой study переоцениваются на `25` fresh seeds.

Это **не новая оптимизация**: параметры фиксируются, новые параметры не ищутся. Confirmation нужен только для проверки seed-robustness top trials.

Для UCB confirmation не запускается, потому что UCB в текущей реализации детерминированный.


In [ ]:
confirmation_rows = []
confirmation_seed_rows = []
confirmation_symbol_frames = []

# Lookup to recover feature_pair metadata by algorithm/method/set.
feature_pair_lookup = {
    (fp["algorithm"], fp["method_group"], fp["set_name"]): fp
    for fp in FEATURE_PAIRS
}


def bandit_params_from_trial_result_row(row: pd.Series) -> dict:
    """Restore executable bandit params from saved trial_results row."""
    algorithm = row["bandit_name"]
    params = {
        "bandit_type": BANDIT_TYPE_MAP[algorithm],
        "lambda_prior": float(row["param_lambda_prior"]),
        "reward_clip": REWARD_CLIP,
    }
    if algorithm in DISCOUNTED_BANDITS:
        params["discount_factor"] = float(row["param_discount_factor"])
    else:
        params["window_size"] = int(row["param_window_size"])

    if algorithm in TS_BANDITS:
        params["noise_std"] = float(row["param_noise_std"])
    else:
        params["ucb_alpha"] = float(row["param_ucb_alpha"])
    return params


if RUN_TS_CONFIRMATION:
    ts_ranking = final_ranking[final_ranking["bandit_name"].isin(TS_BANDITS)].copy()

    if ts_ranking.empty:
        print("No TS studies in this RUN_PRESET; confirmation skipped.")
    else:
        for (algorithm, method_group, set_name), part in ts_ranking.groupby(["bandit_name", "method_group", "set_name"], sort=True):
            top_trials = part.sort_values(
                ranking_cols,
                ascending=[False, False, False, False, True, False, False],
            ).head(TOP_N_CONFIRMATION).copy()

            feature_pair = feature_pair_lookup[(algorithm, method_group, set_name)]

            for confirmation_rank, (_, trial_row) in enumerate(top_trials.iterrows(), start=1):
                original_trial_number = int(trial_row["trial_number"])
                original_objective = float(trial_row["objective_score"])
                bandit_params = bandit_params_from_trial_result_row(trial_row)

                print("=" * 120)
                print(
                    f"TS confirmation: {algorithm} | {method_group} | {set_name} | "
                    f"original trial={original_trial_number} | rank={confirmation_rank}/{TOP_N_CONFIRMATION}"
                )
                print("params:", bandit_params)
                print("confirmation seeds:", CONFIRMATION_SEEDS)
                print("=" * 120)

                seed_summaries = []
                symbol_metric_frames = []

                for seed in CONFIRMATION_SEEDS:
                    symbol_metrics, seed_summary = run_backtest_for_seed(
                        algorithm=algorithm,
                        feature_pair=feature_pair,
                        bandit_params=bandit_params,
                        seed=int(seed),
                    )
                    symbol_metrics.insert(0, "confirmation_seed", int(seed))
                    symbol_metrics.insert(0, "original_trial_number", original_trial_number)
                    symbol_metrics.insert(0, "confirmation_rank_from_optuna", confirmation_rank)
                    symbol_metrics.insert(0, "set_name", set_name)
                    symbol_metrics.insert(0, "method_group", method_group)
                    symbol_metrics.insert(0, "bandit_name", algorithm)
                    symbol_metric_frames.append(symbol_metrics)

                    seed_summary.update({
                        "confirmation_seed": int(seed),
                        "original_trial_number": original_trial_number,
                        "confirmation_rank_from_optuna": confirmation_rank,
                        "bandit_name": algorithm,
                        "method_group": method_group,
                        "set_name": set_name,
                        "z_window": feature_pair["z_window"],
                    })
                    seed_summaries.append(seed_summary)

                agg = aggregate_seed_summaries(seed_summaries)
                confirmation_score, constraint_details = constrained_median_objective_from_agg(agg)

                row = {
                    "bandit_name": algorithm,
                    "method_group": method_group,
                    "set_name": set_name,
                    "z_window": feature_pair["z_window"],
                    "alpha_out": feature_pair["alpha_out"],
                    "original_trial_number": original_trial_number,
                    "confirmation_rank_from_optuna": confirmation_rank,
                    "original_objective_score": original_objective,
                    "confirmation_objective_score": confirmation_score,
                    "n_confirmation_seeds": len(CONFIRMATION_SEEDS),
                    "confirmation_seeds": "|".join(str(s) for s in CONFIRMATION_SEEDS),
                    **constraint_details,
                    **{f"confirmation_{k}": v for k, v in agg.items()},
                    **{f"param_{k}": v for k, v in bandit_params.items() if k not in {"actions"}},
                }
                confirmation_rows.append(row)
                confirmation_seed_rows.extend(seed_summaries)
                if symbol_metric_frames:
                    confirmation_symbol_frames.append(pd.concat(symbol_metric_frames, ignore_index=True))

                pd.DataFrame(confirmation_rows).to_csv(OUTPUT_DIR / "ts_top3_confirmation_results.csv", index=False)
                pd.DataFrame(confirmation_seed_rows).to_csv(OUTPUT_DIR / "ts_top3_confirmation_seed_level.csv", index=False)
                if confirmation_symbol_frames:
                    pd.concat(confirmation_symbol_frames, ignore_index=True).to_csv(
                        DIAGNOSTICS_DIR / "ts_top3_confirmation_symbol_metrics.csv",
                        index=False,
                    )

    if confirmation_rows:
        confirmation_df = pd.DataFrame(confirmation_rows)
        confirmation_ranking = (
            confirmation_df
            .sort_values(
                [
                    "bandit_name", "method_group",
                    "valid",
                    "confirmation_objective_score",
                    "confirmation_median_of_seed_median_return_pct",
                    "confirmation_mean_of_seed_median_return_pct",
                    "confirmation_median_max_drawdown_abs",
                    "confirmation_median_profit_factor",
                    "confirmation_median_total_trades",
                ],
                ascending=[True, True, False, False, False, False, True, False, False],
            )
            .reset_index(drop=True)
        )
        confirmation_ranking["confirmation_rank_within_algorithm_method"] = (
            confirmation_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1
        )
        confirmation_best_per_pair = confirmation_ranking[
            confirmation_ranking["confirmation_rank_within_algorithm_method"].eq(1)
        ].copy()

        confirmation_ranking.to_csv(OUTPUT_DIR / "ts_top3_confirmation_ranking.csv", index=False)
        confirmation_best_per_pair.to_csv(OUTPUT_DIR / "ts_best_trial_per_algorithm_method_after_confirmation.csv", index=False)

        # Unified final selection table: TS uses confirmation winner; UCB uses Optuna winner.
        final_selection_rows = []
        for _, row in best_per_pair.iterrows():
            algorithm = row["bandit_name"]
            method_group = row["method_group"]
            if algorithm in TS_BANDITS:
                match = confirmation_best_per_pair[
                    (confirmation_best_per_pair["bandit_name"].eq(algorithm))
                    & (confirmation_best_per_pair["method_group"].eq(method_group))
                ]
                if not match.empty:
                    conf = match.iloc[0]
                    final_selection_rows.append({
                        "selection_source": "ts_top3_25seed_confirmation",
                        "bandit_name": algorithm,
                        "method_group": method_group,
                        "set_name": conf["set_name"],
                        "selected_trial_number": int(conf["original_trial_number"]),
                        "selected_objective_score": float(conf["confirmation_objective_score"]),
                        "selected_valid": bool(conf["valid"]),
                        "selected_median_of_seed_median_return_pct": float(conf["confirmation_median_of_seed_median_return_pct"]),
                        "selected_mean_of_seed_median_return_pct": float(conf["confirmation_mean_of_seed_median_return_pct"]),
                        "selected_std_of_seed_median_return_pct": float(conf["confirmation_std_of_seed_median_return_pct"]),
                        "selected_median_max_drawdown_abs": float(conf["confirmation_median_max_drawdown_abs"]),
                        "selected_median_profit_factor": float(conf["confirmation_median_profit_factor"]),
                        "selected_median_total_trades": float(conf["confirmation_median_total_trades"]),
                        "n_selection_seeds": int(conf["n_confirmation_seeds"]),
                        "original_optuna_objective_score": float(conf["original_objective_score"]),
                    })
                    continue

            # UCB or fallback if confirmation missing.
            final_selection_rows.append({
                "selection_source": "optuna_5seed_or_ucb",
                "bandit_name": algorithm,
                "method_group": method_group,
                "set_name": row["set_name"],
                "selected_trial_number": int(row["trial_number"]),
                "selected_objective_score": float(row["objective_score"]),
                "selected_valid": bool(row["valid"]),
                "selected_median_of_seed_median_return_pct": float(row["median_of_seed_median_return_pct"]),
                "selected_mean_of_seed_median_return_pct": float(row["mean_of_seed_median_return_pct"]),
                "selected_std_of_seed_median_return_pct": float(row.get("std_of_seed_median_return_pct", 0.0)),
                "selected_median_max_drawdown_abs": float(row["median_max_drawdown_abs"]),
                "selected_median_profit_factor": float(row["median_profit_factor"]),
                "selected_median_total_trades": float(row["median_total_trades"]),
                "n_selection_seeds": int(row["n_seeds"]),
                "original_optuna_objective_score": float(row["objective_score"]),
            })

        final_selection = pd.DataFrame(final_selection_rows)
        final_selection.to_csv(OUTPUT_DIR / "best_trial_per_algorithm_method_with_ts_confirmation.csv", index=False)

        display(confirmation_best_per_pair[[
            "bandit_name", "method_group", "set_name", "original_trial_number",
            "confirmation_objective_score", "valid",
            "confirmation_median_of_seed_median_return_pct",
            "confirmation_mean_of_seed_median_return_pct",
            "confirmation_std_of_seed_median_return_pct",
            "confirmation_median_max_drawdown_abs",
            "confirmation_median_profit_factor",
            "confirmation_median_total_trades",
        ]])

        display(final_selection)
else:
    print("RUN_TS_CONFIRMATION=False; confirmation skipped.")


## 11. Optional: merge outputs from two computers

После завершения двух запусков скопируй папку с другого компьютера внутрь `stage1_optuna_constrained_median_with_ts_confirmation_local`, затем выполни эту ячейку.

Эта ячейка объединяет основные `trial_results_all.csv`. Confirmation-файлы можно объединять отдельно через `ts_top3_confirmation_results.csv`, если оба запуска уже завершены.


In [ ]:
MERGE_ALL_LOCAL_RUNS = False

if MERGE_ALL_LOCAL_RUNS:
    merged_out_dir = OUTPUT_ROOT / "_merged_all_runs"
    merged_out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Merge all main Optuna trials.
    all_trial_files = sorted(OUTPUT_ROOT.glob("*/trial_results_all.csv"))
    print("Found trial files:", len(all_trial_files))
    if all_trial_files:
        merged = pd.concat([pd.read_csv(p) for p in all_trial_files], ignore_index=True)
        merged.to_csv(merged_out_dir / "merged_trial_results_all.csv", index=False)

        merged_ranking = (
            merged
            .sort_values(
                ["bandit_name", "method_group", *ranking_cols],
                ascending=[True, True, False, False, False, False, True, False, False],
            )
            .reset_index(drop=True)
        )
        merged_ranking["rank_within_algorithm_method"] = (
            merged_ranking.groupby(["bandit_name", "method_group"]).cumcount() + 1
        )
        merged_best_per_pair = merged_ranking[merged_ranking["rank_within_algorithm_method"].eq(1)].copy()
        merged_best_per_pair.to_csv(merged_out_dir / "merged_best_trial_per_algorithm_method.csv", index=False)

        display(merged_best_per_pair[[
            "bandit_name", "method_group", "set_name", "trial_number", "valid",
            "median_of_seed_median_return_pct", "mean_of_seed_median_return_pct",
            "median_max_drawdown_abs", "median_profit_factor", "median_total_trades",
            "objective_score",
        ]])

    # 2) Merge TS confirmation outputs, if present.
    confirmation_files = sorted(OUTPUT_ROOT.glob("*/ts_top3_confirmation_results.csv"))
    print("Found TS confirmation files:", len(confirmation_files))
    if confirmation_files:
        merged_confirmation = pd.concat([pd.read_csv(p) for p in confirmation_files], ignore_index=True)
        merged_confirmation.to_csv(merged_out_dir / "merged_ts_top3_confirmation_results.csv", index=False)

    # 3) Merge final selected rows: TS rows use confirmation; UCB rows use Optuna.
    final_selection_files = sorted(OUTPUT_ROOT.glob("*/best_trial_per_algorithm_method_with_ts_confirmation.csv"))
    print("Found final selection files:", len(final_selection_files))
    if final_selection_files:
        merged_final_selection = pd.concat([pd.read_csv(p) for p in final_selection_files], ignore_index=True)
        merged_final_selection.to_csv(merged_out_dir / "merged_best_trial_per_algorithm_method_with_ts_confirmation.csv", index=False)
        display(merged_final_selection)
